# Clinical cohorts in Colab Enterprise

A starter notebook for exploring **CMS Synthetic Patient Data in OMOP Common Data Model** directly in BigQuery, then handing off to the **Data Science Agent**.

**Dataset:** `bigquery-public-data.cms_synthetic_patient_data_omop.*` — 2.3M synthetic patients, fully public, no credentialing required.

Run cells 2-5 in order. Then open the **Data Science Agent** panel on the right and try the prompts at the bottom.

In [ ]:
import os
from google.cloud import bigquery

# In Colab Enterprise this is set automatically to your Qwiklabs project.
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "<YOUR_QWIKLABS_PROJECT_ID>")
client = bigquery.Client(project=PROJECT_ID)

def q(sql):
    """Run a BigQuery SQL string and return a pandas DataFrame."""
    return client.query(sql).to_dataframe()

print(f"Billing project: {client.project}")

In [ ]:
# Smoke test — confirm BigQuery reaches the public OMOP synthetic dataset.
result = q("""
    SELECT COUNT(*) AS n_persons
    FROM `bigquery-public-data.cms_synthetic_patient_data_omop.person`
""")
print(f"Synthetic CMS persons: {int(result['n_persons'][0]):,}")

## Quick dataset tour

The OMOP Common Data Model splits patient data into well-known tables. The big ones in this dataset:

- `person` — demographics (year_of_birth, gender_concept_id, race_concept_id)
- `condition_occurrence` — diagnoses (joined to `concept` for human-readable names)
- `drug_exposure` — prescriptions and administrations
- `procedure_occurrence` — procedures
- `death` — mortality
- `concept` — vocabulary lookup; join to any `*_concept_id` column

Let's pull the top 10 most-recorded conditions:

In [ ]:
top_conditions = q("""
    SELECT c.concept_name, COUNT(*) AS n
    FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence` co
    JOIN `bigquery-public-data.cms_synthetic_patient_data_omop.concept` c
      ON co.condition_concept_id = c.concept_id
    GROUP BY c.concept_name
    ORDER BY n DESC
    LIMIT 10
""")
top_conditions

## Hand off to the Data Science Agent

Open the **Data Science Agent** panel on the right of Colab Enterprise. Try these prompts:

> *"What's the age distribution of patients with type 2 diabetes?"*

> *"Show me the top 10 most-prescribed drugs and the most-common conditions they're prescribed alongside."*

> *"Build a cohort of patients with both diabetes and chronic kidney disease, then plot their condition counts."*

The agent will generate Python that uses the `client` and `q()` helper above. Edit and re-run any of its cells.

## Going further

- The OMOP vocabulary tables (`concept`, `concept_relationship`, `concept_ancestor`, `vocabulary`) make complex condition cohorts straightforward — search by SNOMED, RxNorm, ICD code, or hierarchy.
- For real (non-synthetic) clinical data, MIMIC-IV is at `physionet-data.mimiciv_hosp.*` — requires PhysioNet credentialed access (CITI training + DUA, ~2-4 weeks) and a one-time link of your Google identity at [physionet.org/settings/cloud/](https://physionet.org/settings/cloud/).
- Imaging Data Commons lives at `bigquery-public-data.idc_current.*` — no credentialing required.